# PrimeVarClass - Pipeline Cloud (Google Colab)
Este notebook resolve o problema de falta de espaço no disco local.
Ele faz o download do dataset AlphaMissense diretamente para a memória do Google Cloud, filtra as variantes apenas dos genes BRCA1 e BRCA2 na hora, e salva um CSV leve de poucos megabytes direto no seu Google Drive.

**Instruções:**
1. Clique em 'Conectar' no Colab.
2. Execute a primeira célula para montar o seu Google Drive.
3. Execute a segunda célula para puxar e filtrar os dados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import urllib.request
import os

# 1. Download do arquivo gzip do Zenodo/DeepMind para a RAM do Colab (Ultra Rápido)
url = 'https://zenodo.org/records/8208688/files/AlphaMissense_hg38.tsv.gz'
filename = '/content/AlphaMissense_hg38.tsv.gz'
print('Baixando AlphaMissense para o Colab...')
urllib.request.urlretrieve(url, filename)

# 2. Processamento On-the-Fly
print('Processando os 20GB on-the-fly. Extraindo apenas BRCA1 (chr17) e BRCA2 (chr13)...')
chunks = pd.read_csv(filename, sep='\t', compression='gzip', chunksize=1000000, comment='#')

brca_dfs = []
for i, chunk in enumerate(chunks):
    # Foca nas coordenadas genomicas exatas (GRCh38) do BRCA1 e BRCA2 para evitar Data Leakage
    brca1_mask = (chunk['#CHROM'] == 'chr17') & (chunk['POS'] >= 43044295) & (chunk['POS'] <= 43125483)
    brca2_mask = (chunk['#CHROM'] == 'chr13') & (chunk['POS'] >= 32315086) & (chunk['POS'] <= 32400266)
    brca_chunk = chunk[brca1_mask | brca2_mask]
    if not brca_chunk.empty:
        brca_dfs.append(brca_chunk)
    print(f'Lote {i+1} processado. Variantes BRCA encontradas até agora: {sum(len(df) for df in brca_dfs)}')

final_df = pd.concat(brca_dfs)

# 3. Salva no Google Drive de forma compacta
dest_dir = '/content/drive/MyDrive/IA_dos_numeros_primos_Cloud'
os.makedirs(dest_dir, exist_ok=True)
dest_path = os.path.join(dest_dir, 'AlphaMissense_BRCA_filtered.csv')
final_df.to_csv(dest_path, index=False)
print(f'\nSUCESSO! Arquivo processado e salvo em: {dest_path}')
print('Você já pode rodar os treinamentos do PrimeVarClass localmente com este arquivo leve!')